In [1]:
import torch
import torch.nn as nn

# optuna or sigopt
# asha scheduler
# tune batch size, learning rate

In [2]:
device = torch.device("cpu")
if (torch.cuda.is_available()):
    device = torch.device("cuda")
if (torch.backends.mps.is_available()):
    device = torch.device("mps")

print('Using device:', device)
LEARNING_RATE = 0.0001
NUM_EPOCHS = 1000

Using device: mps


In [3]:
import torchvision
from torchvision import transforms, datasets
from torchvision.datasets import MNIST
from torch.utils.data import DataLoader
# load data


train_transform = transforms.Compose(
    [torchvision.transforms.ToTensor(), torch.flatten])
test_transform = transforms.Compose(
    [torchvision.transforms.ToTensor(), torch.flatten])

train_data = MNIST(root='data', train=True, download=True,
                   transform=train_transform)
test_data = MNIST(root='data', train=False,
                  download=True, transform=test_transform)



In [4]:
class VAE(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(784, 256),
            nn.LeakyReLU(0.2),
            nn.Linear(256, 128),
            nn.LeakyReLU(0.2),
            nn.Linear(128, 64),
            nn.LeakyReLU(0.2),
        )

        self.z_mean = nn.Linear(64, 32)
        self.z_log_var = nn.Linear(64, 32)

        self.decoder = nn.Sequential(
            nn.Linear(32, 64),
            nn.LeakyReLU(0.2),
            nn.Linear(64, 128),
            nn.LeakyReLU(0.2),
            nn.Linear(128, 256),
            nn.LeakyReLU(0.2),
            nn.Linear(256, 784),
            nn.Sigmoid(),
        )

    def reparameterize(self, z_mu, z_log_var):
        esp = torch.randn(z_mu.shape[0], z_mu.shape[1]).to(device)
        z = z_mu + esp * torch.exp(z_log_var/2.)
        return z

    def encoding(self, x):
        x = self.encoder(x)
        z_mean, z_log_var = self.z_mean(x), self.z_log_var(x)
        encoded = self.reparameterize(z_mean, z_log_var)
        return encoded

    def forward(self, x):
        x = self.encoder(x)
        z_mean, z_log_var = self.z_mean(x), self.z_log_var(x)
        encoded = self.reparameterize(z_mean, z_log_var)
        decoded = self.decoder(encoded)
        return encoded, z_mean, z_log_var, decoded

In [5]:
import torch.nn.functional as Functional
from ray import train
def trainable(config):
    train_loader = DataLoader(
        train_data, batch_size=config["batch_size"], shuffle=True)
    test_loader = DataLoader(test_data, batch_size=config["batch_size"], shuffle=False)
    model = VAE()
    model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=config["lr"])
    losses = []
    for epoch in range(50):
        total_loss = 0
        for batch_idx, (features, _) in enumerate(train_loader):
            features = features.to(device)

            encoded, z_mean, z_log_var, decoded = model(features)
            kl_div = -0.5 * torch.sum(1 + z_log_var  - z_mean**2  - torch.exp(z_log_var), axis=1) # sum over latent dimension
            batchsize = kl_div.size(0)
            kl_div = torch.mean(kl_div)

            reconstruction_loss = Functional.mse_loss(decoded, features, reduction='none') # sum over all pixels
            reconstruction_loss = reconstruction_loss.view(batchsize, -1).sum(axis=1)
            reconstruction_loss = reconstruction_loss.mean()

            loss = kl_div + reconstruction_loss
            loss = loss / batchsize
            total_loss += loss
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
        train.report({"losses": total_loss.item()/len(train_loader)})

In [6]:
from ray import tune
from ray.tune.search.optuna import OptunaSearch
from ray.tune.search import ConcurrencyLimiter
from ray.tune.schedulers import ASHAScheduler

# setup search algo
algo = OptunaSearch()
algo = ConcurrencyLimiter(algo, max_concurrent=4)

search_space = {
        "lr": tune.loguniform(1e-5, 1e-1),
        "batch_size": tune.randint(32, 512),
        # "epochs": tune.randint(10, 1000)
    }

    # setup scheduler 
asha_scheduler = ASHAScheduler(
        max_t=1000,
        grace_period=1,
        reduction_factor=2)

tuner = tune.Tuner(
        trainable,
        tune_config=tune.TuneConfig(
            metric="losses",
            mode="min",
            search_alg=algo,
            scheduler=asha_scheduler,
            num_samples=10,
        ),
        param_space=search_space,
        
    )
results = tuner.fit()

2023-11-25 23:14:31,886	WARNING worker.py:2074 -- Warning: The actor ImplicitFunc is very large (52 MiB). Check that its definition is not implicitly capturing a large array or other object in scope. Tip: use ray.put() to put large objects in the Ray object store.
/opt/homebrew/Caskroom/miniforge/base/lib/python3.10/site-packages/numpy/lib/nanfunctions.py:1384: RuntimeWarning: All-NaN slice encountered
  return _nanquantile_unchecked(
2023-11-25 23:18:08,774	WARNING optuna_search.py:552 -- The value nan is not acceptable
2023-11-25 23:18:31,775	WARNING optuna_search.py:552 -- The value nan is not acceptable
2023-11-25 23:19:21,929	INFO tune.py:1047 -- Total run time: 290.70 seconds (290.59 seconds for the tuning loop).


In [8]:
print("Best config: ", results.get_best_result(metric="losses", mode="min"))

Best config:  Result(
  metrics={'losses': 0.05816658207627594},
  path='/Users/boo/ray_results/trainable_2023-11-25_23-14-29/trainable_4ded8b4a_7_batch_size=494,lr=0.0046_2023-11-25_23-16-02',
  filesystem='local',
  checkpoint=None
)
